In [ ]:
import textwrap
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

RANDOM_STATE = 42
TOP_N_GENES = 10
LTS_COLOR = "#80bcc8"
HTS_COLOR = "#d88f91"


def get_code_dir() -> Path:
    """Return the script directory or the current working directory in notebooks."""
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


def get_available_font() -> str:
    """Return the first available font from a preferred list."""
    candidates = [
        "Arial",
        "DejaVu Sans",
        "Liberation Sans",
        "Nimbus Sans",
        "sans-serif",
    ]
    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font in candidates:
        if font in installed_fonts:
            return font
    return "DejaVu Sans"


PLOT_FONT = get_available_font()
plt.rcParams["font.family"] = PLOT_FONT
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150


def ensure_dir(path: Path) -> None:
    """Create a directory and all missing parent directories."""
    Path(path).mkdir(parents=True, exist_ok=True)


def save_figure(fig: plt.Figure, save_path: Path, dpi: int = 300) -> None:
    """Save and close a Matplotlib figure."""
    save_path = Path(save_path)
    fig.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )
    plt.close(fig)


def save_confusion_matrix(
    cm: np.ndarray,
    class_names: list[str],
    save_path: Path,
) -> None:
    """Save a row-colored confusion matrix."""
    fig, ax = plt.subplots(figsize=(5, 4))

    color_grid = np.zeros_like(cm, dtype=float)
    for row_index in range(cm.shape[0]):
        color_grid[row_index, :] = row_index

    cmap = ListedColormap([LTS_COLOR, HTS_COLOR])
    ax.imshow(color_grid, cmap=cmap, vmin=0, vmax=1)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title="Gradient Boosting Test Confusion Matrix",
    )

    for row_index in range(cm.shape[0]):
        for column_index in range(cm.shape[1]):
            ax.text(
                column_index,
                row_index,
                str(cm[row_index, column_index]),
                ha="center",
                va="center",
                color="black",
                fontsize=14,
                fontweight="bold",
            )

    ax.set_xticks(np.arange(-0.5, cm.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, cm.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_roc_curve(y_true: pd.Series, y_prob: np.ndarray, save_path: Path) -> None:
    """Save the test-set ROC curve."""
    auc_value = roc_auc_score(y_true, y_prob)
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"ROC AUC = {auc_value:.4f}",
    )
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Gradient Boosting Test ROC Curve")
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_pr_curve(y_true: pd.Series, y_prob: np.ndarray, save_path: Path) -> None:
    """Save the test-set precision-recall curve."""
    average_precision = average_precision_score(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(
        recall,
        precision,
        linewidth=2,
        label=f"AP = {average_precision:.4f}",
    )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Gradient Boosting Test Precision-Recall Curve")
    ax.legend(loc="lower left")
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_test_score_distribution(
    y_true: pd.Series,
    y_prob: np.ndarray,
    save_path: Path,
) -> None:
    """Save the distribution of predicted HTS probabilities in the test set."""
    y_true_array = np.asarray(y_true)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.hist(
        y_prob[y_true_array == 0],
        bins=30,
        alpha=0.7,
        label="LTS (0)",
        edgecolor="black",
    )
    ax.hist(
        y_prob[y_true_array == 1],
        bins=30,
        alpha=0.7,
        label="HTS (1)",
        edgecolor="black",
    )
    ax.set_xlabel("Predicted probability of HTS")
    ax.set_ylabel("Cell count")
    ax.set_title("Gradient Boosting Test Score Distribution")
    ax.legend()
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_mean_abs_shap_dot(
    shap_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_GENES,
) -> None:
    """Save a dot plot of the highest-ranking genes by mean absolute SHAP value."""
    plot_df = (
        shap_df.head(top_n)
        .sort_values("mean_abs_shap", ascending=True)
        .reset_index(drop=True)
    )

    values = plot_df["mean_abs_shap"].to_numpy()
    labels = [textwrap.fill(str(value), width=30) for value in plot_df["gene"]]
    y_positions = np.arange(len(plot_df))

    cmap = LinearSegmentedColormap.from_list(
        "custom_gradient",
        ["#39489f", "#39bbec", "#f9ed36", "#f38466", "#b81f25"],
    )

    value_min = float(values.min())
    value_max = float(values.max())
    if value_max - value_min < 1e-12:
        norm = Normalize(vmin=value_min - 0.5, vmax=value_max + 0.5)
        sizes = np.full(values.shape, 220.0)
    else:
        norm = Normalize(vmin=value_min, vmax=value_max)
        sizes = 120 + 420 * (values - value_min) / (value_max - value_min)

    figure_height = max(5, 0.42 * len(plot_df))
    fig, ax = plt.subplots(figsize=(7.2, figure_height))

    scatter = ax.scatter(
        values,
        y_positions,
        c=values,
        s=sizes,
        cmap=cmap,
        norm=norm,
        marker="o",
        edgecolors="black",
        linewidths=1,
        alpha=1,
        zorder=3,
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=12)
    ax.set_xlim(0, max(value_max * 1.10, 0.02))
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {len(plot_df)} Genes by SHAP Importance")
    ax.grid(False)
    ax.tick_params(axis="x", length=0, labelsize=11)
    ax.tick_params(axis="y", length=0)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1)

    colorbar = fig.colorbar(scatter, ax=ax, pad=0.02)
    colorbar.set_label("Mean |SHAP value|", rotation=270, labelpad=18, fontsize=10)
    colorbar.ax.tick_params(labelsize=9)

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_mean_abs_shap_bar(
    shap_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_GENES,
) -> None:
    """Save a bar plot of the highest-ranking genes by mean absolute SHAP value."""
    plot_df = (
        shap_df.head(top_n)
        .sort_values("mean_abs_shap", ascending=True)
        .reset_index(drop=True)
    )

    values = plot_df["mean_abs_shap"].to_numpy()
    labels = plot_df["gene"].tolist()

    cmap = LinearSegmentedColormap.from_list(
        "custom_gradient",
        ["#39489f", "#39bbec", "#f9ed36", "#f38466", "#b81f25"],
    )

    value_min = float(values.min())
    value_max = float(values.max())
    if value_max - value_min < 1e-12:
        norm = Normalize(vmin=value_min - 0.5, vmax=value_max + 0.5)
    else:
        norm = Normalize(vmin=value_min, vmax=value_max)

    colors = cmap(norm(values))
    figure_width = max(10, 0.7 * len(plot_df))
    fig, ax = plt.subplots(figsize=(figure_width, 5.0))

    ax.bar(np.arange(len(plot_df)), values, color=colors, width=0.82)
    ax.set_xticks(np.arange(len(plot_df)))
    ax.set_xticklabels(labels, rotation=90, ha="center", fontsize=9)
    ax.set_ylabel("Mean |SHAP value|", fontsize=12)
    ax.set_title("Gradient Boosting SHAP Importance", loc="left", fontsize=12)
    ax.grid(False)
    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0, labelsize=10)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1)

    scalar_mappable = ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, ax=ax, pad=0.02)
    colorbar.set_label("Mean |SHAP value|", rotation=270, labelpad=18, fontsize=10)
    colorbar.ax.tick_params(labelsize=9)

    fig.tight_layout()
    save_figure(fig, save_path)


def extract_shap_array(shap_values: object, n_features: int) -> np.ndarray:
    """Convert common SHAP output formats into a two-dimensional class-1 array."""
    if isinstance(shap_values, list):
        if len(shap_values) > 1:
            return np.asarray(shap_values[1])
        return np.asarray(shap_values[0])

    array = np.asarray(shap_values)

    if array.ndim == 2:
        return array

    if array.ndim == 3:
        if array.shape[1] == n_features and array.shape[2] == 2:
            return array[:, :, 1]
        if array.shape[1] == 2 and array.shape[2] == n_features:
            return array[:, 1, :]

    raise ValueError(f"Unexpected SHAP output shape: {array.shape}")


def load_dataset(data_path: Path) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """Load the expression matrix and validate binary labels."""
    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_path}")

    dataframe = pd.read_csv(data_path)
    if "label" not in dataframe.columns:
        raise ValueError("The dataset must contain a 'label' column.")

    non_feature_columns = ["label"]
    first_column = dataframe.columns[0]
    if first_column != "label" and not pd.api.types.is_numeric_dtype(dataframe[first_column]):
        non_feature_columns.append(first_column)

    feature_columns = [
        column for column in dataframe.columns if column not in non_feature_columns
    ]
    if not feature_columns:
        raise ValueError("No feature columns were detected in the dataset.")

    numeric_features = dataframe[feature_columns].apply(pd.to_numeric, errors="coerce")
    missing_count = int(numeric_features.isna().sum().sum())
    if missing_count > 0:
        print(
            f"[WARNING] {missing_count} missing or non-numeric feature values "
            "were replaced with 0."
        )

    X = numeric_features.fillna(0.0).astype(np.float32)
    y = pd.to_numeric(dataframe["label"], errors="coerce")

    if y.isna().any() or not set(y.unique()).issubset({0, 1}):
        raise ValueError("The 'label' column must contain only 0 and 1.")

    return X, y.astype(int), feature_columns


def tune_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_validation: pd.DataFrame,
    y_validation: pd.Series,
) -> tuple[dict, float, pd.DataFrame]:
    """Select gradient boosting hyperparameters by validation ROC AUC."""
    parameter_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.01, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0],
    }

    best_parameters = None
    best_validation_auc = -np.inf
    tuning_records = []

    print("[INFO] Starting gradient boosting hyperparameter tuning...")

    for parameters in ParameterGrid(parameter_grid):
        model = GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **parameters,
        )
        model.fit(X_train, y_train)

        validation_probability = model.predict_proba(X_validation)[:, 1]
        validation_prediction = (validation_probability >= 0.5).astype(int)

        validation_auc = roc_auc_score(y_validation, validation_probability)
        validation_accuracy = accuracy_score(y_validation, validation_prediction)
        validation_f1 = f1_score(y_validation, validation_prediction, zero_division=0)
        validation_precision = precision_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        )
        validation_recall = recall_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        )

        tuning_records.append(
            {
                **parameters,
                "validation_auc": validation_auc,
                "validation_accuracy": validation_accuracy,
                "validation_f1": validation_f1,
                "validation_precision": validation_precision,
                "validation_recall": validation_recall,
            }
        )

        print(
            f"[INFO] n_estimators={parameters['n_estimators']}, "
            f"learning_rate={parameters['learning_rate']}, "
            f"max_depth={parameters['max_depth']}, "
            f"subsample={parameters['subsample']} | "
            f"Validation AUC={validation_auc:.4f}"
        )

        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            best_parameters = parameters.copy()

    if best_parameters is None:
        raise RuntimeError("Hyperparameter tuning did not produce a valid model.")

    tuning_results = pd.DataFrame(tuning_records).sort_values(
        "validation_auc",
        ascending=False,
    )

    return best_parameters, best_validation_auc, tuning_results


def perform_shap_analysis(
    final_model: GradientBoostingClassifier,
    X_all: pd.DataFrame,
    feature_columns: list[str],
    output_directory: Path,
    figure_directory: Path,
) -> None:
    """Explain the refitted final model and summarize SHAP importance on all cells."""
    import shap

    print("[INFO] Calculating SHAP values for the refitted final model...")

    explainer = shap.TreeExplainer(final_model)
    raw_shap_values = explainer.shap_values(X_all, check_additivity=False)
    shap_array = extract_shap_array(
        raw_shap_values,
        n_features=len(feature_columns),
    )

    shap.summary_plot(
        shap_array,
        X_all,
        feature_names=feature_columns,
        show=False,
    )
    summary_figure = plt.gcf()
    summary_figure.set_size_inches(10, 8)
    save_figure(
        summary_figure,
        figure_directory / "gbm_shap_summary_beeswarm.png",
    )

    mean_absolute_shap = np.abs(shap_array).mean(axis=0)
    mean_shap = shap_array.mean(axis=0)

    shap_importance = pd.DataFrame(
        {
            "gene": feature_columns,
            "mean_abs_shap": mean_absolute_shap,
            "mean_shap": mean_shap,
        }
    ).sort_values("mean_abs_shap", ascending=False)

    shap_importance.to_csv(
        output_directory / "gbm_shap_gene_importance_final_model.csv",
        index=False,
    )
    shap_importance.head(TOP_N_GENES).to_csv(
        output_directory / f"gbm_top{TOP_N_GENES}_shap_genes.csv",
        index=False,
    )

    plot_mean_abs_shap_dot(
        shap_importance,
        figure_directory / f"gbm_top{TOP_N_GENES}_shap_importance_dot.png",
        top_n=TOP_N_GENES,
    )
    plot_mean_abs_shap_bar(
        shap_importance,
        figure_directory / f"gbm_top{TOP_N_GENES}_shap_importance_bar.svg",
        top_n=TOP_N_GENES,
    )


def main() -> None:
    """Run the complete HTS-versus-LTS gradient boosting workflow."""
    code_directory = get_code_dir()
    data_path = code_directory / "df_expr.csv"
    output_directory = code_directory / "gbm_hts_lts_results"
    figure_directory = output_directory / "figures"

    ensure_dir(output_directory)
    ensure_dir(figure_directory)

    print(f"[INFO] Working directory: {code_directory}")
    print(f"[INFO] Using font: {PLOT_FONT}")
    print(f"[INFO] Reading dataset: {data_path}")

    X, y, feature_columns = load_dataset(data_path)

    print(f"[INFO] Feature matrix shape: {X.shape}")
    print(f"[INFO] HTS cells (1): {(y == 1).sum()}")
    print(f"[INFO] LTS cells (0): {(y == 0).sum()}")

    X_temporary, X_test, y_temporary, y_test = train_test_split(
        X,
        y,
        test_size=1 / 3,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    X_train, X_validation, y_train, y_validation = train_test_split(
        X_temporary,
        y_temporary,
        test_size=0.5,
        stratify=y_temporary,
        random_state=RANDOM_STATE,
    )

    print(f"[INFO] Training set size: {X_train.shape[0]}")
    print(f"[INFO] Validation set size: {X_validation.shape[0]}")
    print(f"[INFO] Test set size: {X_test.shape[0]}")

    best_parameters, best_validation_auc, tuning_results = tune_model(
        X_train,
        y_train,
        X_validation,
        y_validation,
    )

    tuning_results.to_csv(
        output_directory / "validation_tuning_results.csv",
        index=False,
    )

    print(f"[INFO] Best parameters: {best_parameters}")
    print(f"[INFO] Best validation ROC AUC: {best_validation_auc:.4f}")

    X_train_validation = pd.concat([X_train, X_validation], axis=0)
    y_train_validation = pd.concat([y_train, y_validation], axis=0)

    final_model = GradientBoostingClassifier(
        random_state=RANDOM_STATE,
        **best_parameters,
    )
    final_model.fit(X_train_validation, y_train_validation)

    print("[INFO] Evaluating the refitted final model on the independent test set...")

    test_probability = final_model.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= 0.5).astype(int)

    test_auc = roc_auc_score(y_test, test_probability)
    test_accuracy = accuracy_score(y_test, test_prediction)
    test_f1 = f1_score(y_test, test_prediction, zero_division=0)
    test_precision = precision_score(y_test, test_prediction, zero_division=0)
    test_recall = recall_score(y_test, test_prediction, zero_division=0)
    test_average_precision = average_precision_score(y_test, test_probability)

    metrics = pd.DataFrame(
        [
            {
                "best_validation_auc": best_validation_auc,
                "test_auc": test_auc,
                "test_accuracy": test_accuracy,
                "test_f1": test_f1,
                "test_precision": test_precision,
                "test_recall": test_recall,
                "test_average_precision": test_average_precision,
                "best_n_estimators": best_parameters["n_estimators"],
                "best_learning_rate": best_parameters["learning_rate"],
                "best_max_depth": best_parameters["max_depth"],
                "best_subsample": best_parameters["subsample"],
            }
        ]
    )
    metrics.to_csv(output_directory / "test_metrics.csv", index=False)

    report = classification_report(
        y_test,
        test_prediction,
        target_names=["LTS", "HTS"],
        digits=4,
        zero_division=0,
    )
    with open(
        output_directory / "classification_report.txt",
        "w",
        encoding="utf-8",
    ) as file_handle:
        file_handle.write(report)

    test_predictions = pd.DataFrame(
        {
            "true_label": y_test.to_numpy(),
            "true_class": y_test.map({0: "LTS", 1: "HTS"}).to_numpy(),
            "pred_label": test_prediction,
            "pred_class": pd.Series(test_prediction).map({0: "LTS", 1: "HTS"}),
            "pred_prob_HTS": test_probability,
        }
    )
    test_predictions.to_csv(
        output_directory / "test_predictions.csv",
        index=False,
    )

    confusion = confusion_matrix(y_test, test_prediction)
    save_confusion_matrix(
        confusion,
        ["LTS", "HTS"],
        figure_directory / "test_confusion_matrix.svg",
    )
    plot_roc_curve(
        y_test,
        test_probability,
        figure_directory / "test_roc_curve.png",
    )
    plot_pr_curve(
        y_test,
        test_probability,
        figure_directory / "test_pr_curve.png",
    )
    plot_test_score_distribution(
        y_test,
        test_probability,
        figure_directory / "test_score_distribution.png",
    )

    feature_importance = pd.DataFrame(
        {
            "gene": feature_columns,
            "importance": final_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    feature_importance.to_csv(
        output_directory / "gbm_feature_importance.csv",
        index=False,
    )

    try:
        perform_shap_analysis(
            final_model,
            X,
            feature_columns,
            output_directory,
            figure_directory,
        )
    except Exception as error:
        raise RuntimeError(f"SHAP analysis failed: {error}") from error

    print("[RESULT] Analysis completed successfully.")
    print(f"[RESULT] Output directory: {output_directory}")
    print(f"[RESULT] Best parameters: {best_parameters}")
    print(f"[RESULT] Best validation ROC AUC: {best_validation_auc:.4f}")
    print(f"[RESULT] Test ROC AUC: {test_auc:.4f}")
    print(f"[RESULT] Test accuracy: {test_accuracy:.4f}")
    print(f"[RESULT] Test F1 score: {test_f1:.4f}")
    print(f"[RESULT] Test precision: {test_precision:.4f}")
    print(f"[RESULT] Test recall: {test_recall:.4f}")
    print(f"[RESULT] Test average precision: {test_average_precision:.4f}")


if __name__ == "__main__":
    main()
